In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, f1_score

import joblib

In [3]:
df = pd.read_csv("crypto_trading_dataset.csv")

In [4]:
df["price_change_pct"] = ((df["close_price"]-df["open_price"]) / df["open_price"])*100

In [5]:
df["MA7"] = df["close_price"].rolling(7).mean()
df["MA30"] = df["close_price"].rolling(30).mean()

In [6]:
df["volatility"] = df["high_price"] - df["low_price"]

In [7]:
df["avg_volume"] = df["volume"].rolling(7).mean()

df["volume_spike"] = np.where(
    df["volume"] > df["avg_volume"],
    1,
    0
)

In [8]:
df["target"] = np.where(
    df["close_price"].shift(-1) > df["close_price"],
    1,
    0
)

In [9]:
df.dropna(inplace=True)

In [10]:
X = df[
[
"price_change_pct",
"MA7",
"MA30",
"volatility",
"volume_spike"
]
]

y = df["target"]

In [11]:
X_train,X_test,y_train,y_test = train_test_split(
X,y,test_size=0.2,random_state=42
)

In [12]:
model = LogisticRegression()

model.fit(X_train,y_train)

pred = model.predict(X_test)

C:\Users\krish\AppData\Roaming\Python\Python314\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [13]:
print("Accuracy:",accuracy_score(y_test,pred))
print("Precision:",precision_score(y_test,pred))
print("F1 Score:",f1_score(y_test,pred))

Accuracy: 0.5035971223021583
Precision: 0.45
F1 Score: 0.34285714285714286


In [14]:
rf=RandomForestClassifier()

rf.fit(X_train,y_train)

pred=rf.predict(X_test)

In [15]:
joblib.dump(rf,"crypto_model.pkl")

['crypto_model.pkl']

In [16]:
model=joblib.load("crypto_model.pkl")

In [21]:
import sys
print(sys.executable)

c:\python314\python.exe


day 2

In [25]:
!pip install fastapi uvicorn
from fastapi import FastAPI
from pydantic import BaseModel
import joblib
import numpy as np

app = FastAPI()

# Load trained model
model = joblib.load("crypto_model.pkl")



[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
  Using cached fastapi-0.136.1-py3-none-any.whl.metadata (28 kB)
  Using cached uvicorn-0.46.0-py3-none-any.whl.metadata (6.7 kB)
  Using cached starlette-1.0.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached pydantic-2.13.3-py3-none-any.whl.metadata (108 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.46.3-cp314-cp314-win_amd64.whl.metadata (6.7 kB)
Using cached fastapi-0.136.1-py3-none-any.whl (117 kB)
Using cached uvicorn-0.46.0-py3-none-any.whl (70 kB)
Using cached annotated_doc-0.0.4-py3-none-any.whl (5.3 kB)
Using cached pydantic-2.13.3-py3-none-any.whl (471 kB)
Using cached pydantic_core-2.46.3-cp314-cp314-win_amd64.whl (2.1 MB)
Using cached annotated_types-0.7.0-py3-none-any.whl (13 kB)
Using cached s

In [26]:
# Input Schema (Validation)
class CryptoInput(BaseModel):
    coin_name:str
    open_price:float
    close_price:float
    high_price:float
    low_price:float
    volume:float


# Feature Engineering Function
def create_features(data):

    price_change_pct=((data.close_price-data.open_price)/
                      data.open_price)*100

    ma7=data.close_price
    ma30=data.close_price

    volatility=data.high_price-data.low_price

    volume_spike=1 if data.volume>100000 else 0

    return [[
        price_change_pct,
        ma7,
        ma30,
        volatility,
        volume_spike
    ]]


# Prediction Endpoint
@app.post("/predict")
def predict(data:CryptoInput):

    try:
        features=create_features(data)

        prediction=model.predict(features)[0]

        confidence=0.72

        if prediction==1:
            trend="Bullish"
        else:
            trend="Bearish"

        if data.high_price-data.low_price>500:
            vol_level="High"
        else:
            vol_level="Medium"

        return {
            "prediction":trend,
            "confidence_score":confidence,
            "volatility_level":vol_level
        }

    except Exception as e:
        raise HTTPException(
            status_code=500,
            detail=str(e)
        )


# Optional Enhancement
@app.get("/market-insight")
def market_insight():

    return {
        "Coin":"Bitcoin",
        "Trend":"Bullish",
        "Volatility":"Medium",
        "Volume Trend":"Increasing"
    }

In [27]:
pip install uvicorn

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip
